In [1]:
import torch
import numpy as np
import torchvision.transforms as tvf

from PIL import Image

from unicorrn.model import build_model
from unicorrn.utils import safe_load_weights, plot_correspondences
from unicorrn.utils.config import read_yaml_config
from fvcore.nn import parameter_count

from unicorrn.inference import (
    coarse_only, 
    coarse_to_fine,
    init_query_points, 
    cycle_uniform_grid_inference
)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
xFormers is available.
Flash attention is not available


/projects/vig/prajnan/envs/unicorrn/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead


In [ ]:
img1_path = "../assets/image_a.png"
img2_path = "../assets/image_b.png"

img1 = np.array(Image.open(img1_path).convert("RGB"))
img2 = np.array(Image.open(img2_path).convert("RGB"))


((1180, 1600, 3), (1036, 1554, 3), dtype('uint8'))

In [ ]:
MODEL_CONFIG_PATH="../configs/models/unicorrn_large_stage2.yml"
CKPT_PATH="../pretrained_models/UniCorrn_Large_Stage2.pth"

model_cfg = read_yaml_config(MODEL_CONFIG_PATH)

model = build_model(model_cfg.NAME, cfg=model_cfg)
print(f"Model params: {parameter_count(model)[''] / 1e6:.1f}M")

weights = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
safe_load_weights(model, weights["model"])

Model params: 599.762178M
weights safely loaded


In [ ]:
H, W = img1.shape[:2]
queries = init_query_points(H, W, grid_size=32).view(-1, 2).numpy()

kpts1, kpts2, confidence, _ = coarse_to_fine(
    img1,
    img2,
    queries,
    model,
    select_layer=-1,
    unified_model=True
)

mask = confidence.squeeze() >= 3.8

kpts1 = kpts1[mask]
kpts2 = kpts2[mask]


plot_correspondences(img1, img2, kpts1, kpts2, marker_size=1, plot_line=False, save_path="example.jpg")
